In [ ]:
import pandas as pd
import numpy as np
import anc2vec


embeddings = anc2vec.get_embeddings() # dizionario <temine_go, vettore_embedding>

# Output Uniprot
AD_nodes_path = '../networks/AD_nodes.csv'
PD_nodes_path = '../networks/PD_nodes.csv'

# Output STRINGdb
AD_edges_path = '../networks/AD_edges.csv'
PD_edges_path = '../networks/PD_edges.csv'

def merge_embeddings(go_list, go_to_embeddings):
    """
    Aggregates embeddings by summing the vectors instead of averaging.
    """
    embeddings = [go_to_embeddings[g.strip()] for g in go_list if g in go_to_embeddings]
    
    if not embeddings:  # If the list is empty, return a null embedding with the same dimension
        return np.zeros_like(next(iter(go_to_embeddings.values()))).tolist()

    return np.sum(embeddings, axis=0).tolist()  # Sum of the vectors


def embed_gos(nodes_path):
    """
    Outputs a dataframe associating each protein (STRING id) with its feature vector (anc2vec embedding).
    """
    nodes_df = pd.read_csv(nodes_path)[['name', 'Gene Ontology IDs']]
    unique_go_ids = set(id.strip() for ids in nodes_df['Gene Ontology IDs'].str.split(';') for id in ids)
    go_to_embeddings = {go:embeddings[go] for go in unique_go_ids if go in embeddings}

    # Aggregates all GO terms for each record
    nodes_df['GO_embeddings'] = nodes_df['Gene Ontology IDs'].apply(
        lambda x: merge_embeddings([go.strip() for go in x.split(';')], go_to_embeddings)
    )

    nodes_df.columns = ['STRING_id', 'Gene Ontology IDs', 'GO_embeddings']
    return nodes_df

def edges_map(edges_path):
    """
    Outputs a dataframe where each pair of proteins (STRING id) represents an edge between them,
    including the 'experimentally_determined_interaction' value.
    """
    raw_edges_df = pd.read_csv(edges_path, usecols=["name", "experimentally_determined_interaction"])

    # Estrae le coppie di proteine dalla colonna 'name'
    edges_df = raw_edges_df["name"].str.extract(r'(\S+) \(interacts with\) (\S+)')
    edges_df.columns = ["node1", "node2"]

    edges_df["experimentally_determined_interaction"] = raw_edges_df["experimentally_determined_interaction"]

    return edges_df


def merge_datasets(df1, df2):
    """
    Combines the two input datasets, assigning label 2 to shared records.
    """
    df_combined = pd.concat([df1, df2], ignore_index=True)
    df_combined["label"] = df_combined.groupby("STRING_id")["label"].transform(lambda x: 2 if len(x) > 1 else x)
    df_final = df_combined.drop_duplicates(subset=["STRING_id"]).reset_index(drop=True)

    return df_final


# 1. Replace GO terms with embeddings
AD_embedded_df = embed_gos(AD_nodes_path)
PD_embedded_df = embed_gos(PD_nodes_path)

# 2. Create edge mappings
AD_edges = edges_map(AD_edges_path)
PD_edges = edges_map(PD_edges_path)

# 3. Apply label (0 = AD, 1 = PD)
AD_embedded_df['label'] = 0
PD_embedded_df['label'] = 1

# 4. Merge AD, PD (common = 2)
merged_embedded_df = merge_datasets(AD_embedded_df, PD_embedded_df)

prot_name_to_id = dict(zip(merged_embedded_df['STRING_id'], merged_embedded_df.index))

AD_edges['node1_id'] = AD_edges['node1'].map(prot_name_to_id)
AD_edges['node2_id'] = AD_edges['node2'].map(prot_name_to_id)
PD_edges['node1_id'] = PD_edges['node1'].map(prot_name_to_id)
PD_edges['node2_id'] = PD_edges['node2'].map(prot_name_to_id)


{'9606.ENSP00000272638': 0,
 '9606.ENSP00000240587': 1,
 '9606.ENSP00000464391': 2,
 '9606.ENSP00000262629': 3,
 '9606.ENSP00000216484': 4,
 '9606.ENSP00000377296': 5,
 '9606.ENSP00000357727': 6,
 '9606.ENSP00000400906': 7,
 '9606.ENSP00000334538': 8,
 '9606.ENSP00000261973': 9,
 '9606.ENSP00000357876': 10,
 '9606.ENSP00000301522': 11,
 '9606.ENSP00000262746': 12,
 '9606.ENSP00000388107': 13,
 '9606.ENSP00000478771': 14,
 '9606.ENSP00000441543': 15,
 '9606.ENSP00000355046': 16,
 '9606.ENSP00000354687': 17,
 '9606.ENSP00000483424': 18,
 '9606.ENSP00000262891': 19,
 '9606.ENSP00000012443': 20,
 '9606.ENSP00000374455': 21,
 '9606.ENSP00000378487': 22,
 '9606.ENSP00000264710': 23,
 '9606.ENSP00000262554': 24,
 '9606.ENSP00000261366': 25,
 '9606.ENSP00000357283': 26,
 '9606.ENSP00000355747': 27,
 '9606.ENSP00000295225': 28,
 '9606.ENSP00000329357': 29,
 '9606.ENSP00000410339': 30,
 '9606.ENSP00000385751': 31,
 '9606.ENSP00000407586': 32,
 '9606.ENSP00000245185': 33,
 '9606.ENSP00000409555':